# Philadelphia — Single-Tax Static Ledger (Tier 1)

## What this notebook answers

Which bundles of abolished Philadelphia city/school taxes on **labor and capital** can be
funded out of full capture of Philadelphia land rent — parcel site rent plus road and curb
rents — and how much capitalization of the abolished taxes back into land rent (`kappa`) does
each bundle need to break even?

Built against `docs/SINGLE_TAX_LEDGER_SPEC.md`. Module: `lvt/single_tax.py`. Tests:
`tests/test_single_tax.py`.

## The math

    Target(B)  = sum(T_j for j in B) + T_land
    Supply(B)  = phi * (1 - h) * (R0 + sum(kappa_j * T_j)) + G
    Pot        = Supply - Target
    kappa*(B)  = (Target - G - phi*(1-h)*R0) / (phi * (1-h) * sum(T_j))

`T_land`, the land portion of today's property tax, is **absorbed, not abolished** — the City
and School District must be made whole for it out of the rent levy, so it enters Target once
for every bundle and never appears in an abolition set. `kappa*` is exact because `kappa`
enters linearly: `kappa* <= 0` means the bundle pencils with no capitalization at all,
`kappa* > 1` means it needs **super-ATCOR** capitalization — more rent than the revenue
foregone. That is not impossible: Gaffney's EBCOR (Excess Burden Comes Out of Rents) holds that
abolishing a *distortionary* tax raises rent by more than the tax, because the excess burden is
recovered too. Do not read `kappa* > 1` as "cannot be done".

Audited twice: a 2026-08-26 self-check and the 2026-08-27 independent adversarial pass
(`analysis/audits/single_tax_ledger_audit_2026-08-27.md`), whose findings are all applied —
including the haircut now multiplying the capitalized increment in the formulas above.

## What this is not

`kappa` is a **swept parameter, not an estimate**. Nothing here models a behavioural response
— no migration, no agglomeration, no supply elasticity, no labour-supply effect. The wage
family's `kappa` band is *anchored* at the top by Jacob & Livas (2026)'s GE counterfactual
(Tier 2, completed 2026-08-28: their Table 4 land-value object turned out to be an annual
floorspace-payment flow, so it maps to `kappa` with no discount-rate bridge — see
`docs/KAPPA_CAPITALIZATION_EVIDENCE.md` section 4), but the behavioural modelling itself
stays in their paper, not here. See spec section 10.

## Section 1 — Imports and Constants

In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.philadelphia import parcel_cache_path, tax_year_params
from lvt.single_tax import (
    BUNDLES,
    G_COMPONENTS,
    G_SCENARIOS,
    KAPPA_BOUNDS,
    KAPPA_SCENARIOS,
    build_ledger,
    bundle_amounts,
    kappa_bounds_frame,
    kappa_star,
    ledger_frame,
    road_rent_frame,
    sweep_bundles,
    validate_ledger,
    vintage_is_consistent,
)
from lvt.ubi_utils import model_full_land_rent_tax

TAX_YEAR = 2026
TY = tax_year_params(TAX_YEAR)
COMBINED_RATE = TY.combined_rate_pct / 100

LEDGER_VINTAGE = 'FY2026'          # 'FY2026' projection (default) or 'FY2025' actual
DISCOUNT_RATES = (0.03, 0.04, 0.05, 0.06, 0.07)
DEFAULT_I = 0.05
PHIS = (1.0, 0.85)
HAIRCUTS = (0.0, 0.05, 0.10, 0.15)
SURFACES = ('opa', 'lycd')
BASES = ('taxable', 'full')

CITY_NAME = f'philadelphia_single_tax_ty{TAX_YEAR}'
REPORT_DIR = REPO_ROOT / 'analysis' / 'reports' / 'philadelphia_single_tax'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('data')
PARCEL_PATH = parcel_cache_path(TAX_YEAR, DATA_DIR)

print(TY.describe())
print(f'Ledger vintage: {LEDGER_VINTAGE} | default i {DEFAULT_I:.0%} | export slug {CITY_NAME}')

TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection)
Ledger vintage: FY2026 | default i 5% | export slug philadelphia_single_tax_ty2026


C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Section 2 — Parcel Data and the Two Land Surfaces

Read-only on the shared year-keyed cache; `model.ipynb` owns building it. The LYCD join is
the one from `model_lvt_ubi.ipynb` Section 3, with the same key normalisation and the same
corrected land-base ratio guard.

**Only `taxable_land_value` is read from the LYCD export.** The current-tax split comes
from the assessor's land alone (Section 4) — an independent land surface answers "how much
rent does this site throw off", never "what is this parcel billed today". That distinction
is the substance of Limitation 19 in `docs/LVT_UBI_GUIDE.md`, fixed 2026-08-26 (PR #30);
this notebook now passes the assessor's land as the baseline and the alternative surface as
the rent basis, with `baseline_total_col` verifying the reconstruction against the billed
total.

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}'
    )
gdf = gpd.read_parquet(PARCEL_PATH)
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
for c in ('taxable_land', 'taxable_building', 'exempt_land'):
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0.0).clip(lower=0.0)

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable land:  ${gdf["taxable_land"].sum()/1e9:.3f}B')
print(f'  exempt land:   ${gdf["exempt_land"].sum()/1e9:.3f}B')

Loaded 583,204 parcels for TY2026
  taxable land:  $43.023B
  exempt land:   $10.722B


In [3]:
# LYCD land surface, joined exactly as model_lvt_ubi.ipynb Section 3 does.
_lycd_path = REPO_ROOT / 'analysis' / 'data' / f'philadelphia_lycd_ty{TAX_YEAR}.csv'
if not _lycd_path.exists():
    raise FileNotFoundError(
        f'{_lycd_path} not found. Run cities/philadelphia/model_lycd.ipynb for '
        f'TY{TAX_YEAR} first.'
    )
_lycd = pd.read_csv(_lycd_path, usecols=['parcel_id', 'taxable_land_value'],
                    dtype={'parcel_id': str})
_dupes = int(_lycd['parcel_id'].duplicated().sum())
_lycd = _lycd.drop_duplicates(subset='parcel_id', keep='first')
_lycd['_key'] = _lycd['parcel_id'].astype(str).str.strip().str.lstrip('0')

gdf['_key'] = gdf['parcel_number'].str.lstrip('0')
gdf = gdf.merge(_lycd[['_key', 'taxable_land_value']], on='_key', how='left')
_matched = gdf['taxable_land_value'].notna().mean()
gdf['land_opa'] = gdf['taxable_land']
gdf['land_lycd'] = gdf['taxable_land_value'].fillna(gdf['taxable_land'])
gdf = gdf.drop(columns=['_key', 'taxable_land_value'])

_ratio = gdf['land_lycd'].sum() / gdf['land_opa'].sum()
print(f'LYCD land joined to {_matched*100:.1f}% of parcels ({_dupes} duplicate ids dropped)')
print(f'LYCD land base: ${gdf["land_lycd"].sum()/1e9:.3f}B '
      f'({_ratio:.3f}x the OPA base of ${gdf["land_opa"].sum()/1e9:.3f}B)')
assert _matched > 0.95, f'only {_matched*100:.1f}% of parcels matched the LYCD export'
# Land-BASE ratio, not the split-rate millage ratio (29.001 vs 23.076 = 1.257, a different
# quantity). See the guard comment in model_lvt_ubi.ipynb Section 3.
assert 1.25 < _ratio < 1.65, f'LYCD/OPA land base ratio {_ratio:.3f} outside 1.25-1.65'

for s in SURFACES:
    gdf[f'full_land_{s}'] = gdf[f'land_{s}'] + gdf['exempt_land']

LYCD land joined to 100.0% of parcels (2 duplicate ids dropped)
LYCD land base: $62.626B (1.456x the OPA base of $43.023B)


## Section 3 — The Rent Supply Side

`R0` is gross annual site rent at full capture: `sum(L * (i + t))`. Computed by calling
`model_full_land_rent_tax` once per (surface, basis, i) rather than reimplementing the
gross-up, so this notebook and the LVT-UBI model can never disagree about what site rent is.

The `full` basis reaches currently-exempt land. It is reported as a sensitivity only — it
would require changing Pennsylvania exemption law and is fiscally circular for City- and
School-owned parcels.

A4 (audit 2026-08-26): the `lycd`/`full` cell is a **hybrid surface** — LYCD taxable land
plus OPA exempt land, because no LYCD-consistent exempt-land surface exists. Read it as
indicative rather than as a LYCD result.

In [4]:
r0_by_case = {}
summaries = {}
for surface in SURFACES:
    for basis in BASES:
        # land_value_col is the BASELINE and is always the assessor's land; the surface
        # the rent is priced from goes in rent_basis_col. Those two roles were conflated
        # before the Limitation-19 fix (PR #30), which is what produced the overstated
        # LYCD baseline. baseline_total_col makes the reconstruction verify itself
        # against the billed total rather than passing silently.
        rent_col = f'full_land_{surface}' if basis == 'full' else f'land_{surface}'
        for i in DISCOUNT_RATES:
            summary, _ = model_full_land_rent_tax(
                gdf,
                land_value_col='land_opa',
                improvement_value_col='taxable_building',
                rent_basis_col=rent_col,
                baseline_total_col='taxable_total',
                discount_rate=i,
                combined_rate=COMBINED_RATE,
                capture_rate=1.0,
                exemption_flag_col='full_exmp',
            )
            r0_by_case[(surface, basis, i)] = summary['total_land_rent']
            summaries[(surface, basis, i)] = summary

_r = pd.Series(r0_by_case).rename('R0')
_r.index.names = ['surface', 'basis', 'i']
print((_r.unstack('i') / 1e9).round(3).to_string())
print('\nR0 in $B, gross annual site rent at capture 1.0')

i                 0.03   0.04   0.05   0.06   0.07
surface basis                                     
lycd    full     2.799  3.436  4.072  4.708  5.344
        taxable  2.755  3.382  4.008  4.634  5.260
opa     full     1.937  2.377  2.817  3.257  3.698
        taxable  1.893  2.323  2.753  3.184  3.614

R0 in $B, gross annual site rent at capture 1.0


In [5]:
# Wiring checks against the executed LVT-UBI notebook.
_opa = r0_by_case[('opa', 'taxable', DEFAULT_I)]
_lycd_r0 = r0_by_case[('lycd', 'taxable', DEFAULT_I)]
assert abs(_opa / 2_753_379_787.0 - 1) < 0.01, f'OPA R0 drift: ${_opa:,.0f}'
assert abs(_lycd_r0 / 4_007_948_272.0 - 1) < 0.01, f'LYCD R0 drift: ${_lycd_r0:,.0f}'

# R0 must be exactly linear in (i + t) -- the gross-up identity.
_L = summaries[('opa', 'taxable', DEFAULT_I)]['total_land_value_rent_basis']
for i in DISCOUNT_RATES:
    assert np.isclose(r0_by_case[('opa', 'taxable', i)], _L * (i + COMBINED_RATE))

print(f'R0 wiring OK. OPA ${_opa/1e9:.3f}B | LYCD ${_lycd_r0/1e9:.3f}B at i={DEFAULT_I:.0%}')

R0 wiring OK. OPA $2.753B | LYCD $4.008B at i=5%


In [6]:
# Population: the dividend denominator, read from the LVT-UBI tract export.
_tracts = pd.read_csv(REPO_ROOT / 'analysis' / 'data' /
                      f'philadelphia_lvt_ubi_ty{TAX_YEAR}.csv')
POPULATION = float(_tracts['total_pop'].sum())
assert abs(POPULATION / 1_593_208.0 - 1) < 0.01, f'population drift: {POPULATION:,.0f}'
print(f'Population (ACS, {len(_tracts)} tracts): {POPULATION:,.0f}')

Population (ACS, 408 tracts): 1,593,208


## Section 4 — The Revenue Ledger

Every line carries its source, vintage, accrual basis and a verified/estimate status; nothing
enters a computation without a source. The two property lines come from the **OPA** run of the
LVT-UBI model, not from budget documents — the budget does not publish a land/building split,
and the baseline must be built on the assessor's land regardless of which surface prices
the rent — the substance of Limitation 19, fixed 2026-08-26 (PR #30).

Vintage is FY2026 current projection, which matches the TY2026 property modelling. That is a
deliberate deviation from the spec's FY2024 default, taken because the spec's own rule is not
to mix vintages: FY2026 is the consistent choice, not FY2024.

In [7]:
_p = summaries[('opa', 'taxable', DEFAULT_I)]
LAND_TAX = _p['total_current_land_tax']
BUILDING_TAX = _p['total_building_tax']
_levy = LAND_TAX + BUILDING_TAX
assert abs(_levy / 2_142_649_826.0 - 1) < 0.01, f'levy drift: ${_levy:,.0f}'

lines = build_ledger(LAND_TAX, BUILDING_TAX, vintage=LEDGER_VINTAGE)
validate_ledger(lines)

print(f'Current TY{TAX_YEAR} combined levy: ${_levy/1e9:.3f}B')
print(f'  land share (ABSORBED):   ${LAND_TAX/1e9:.3f}B')
print(f'  building share (capital):${BUILDING_TAX/1e9:.3f}B')
print(f'\nCross-check: model-implied School District share '
      f'${(_levy - 942_747_555)/1e9:.3f}B vs the City Controller\'s reported '
      f'~$1.2B SDP real estate tax. Consistent.')

Current TY2026 combined levy: $2.143B
  land share (ABSORBED):   $0.602B
  building share (capital):$1.540B

Cross-check: model-implied School District share $1.200B vs the City Controller's reported ~$1.2B SDP real estate tax. Consistent.


In [8]:
_lf = ledger_frame(lines)
_show = _lf[['name', 'amount', 'classification', 'body', 'vintage', 'status']].copy()
_show['amount'] = (_show['amount'] / 1e6).round(1)
_show = _show.rename(columns={'amount': '$M'})
print(_show.to_string(index=False))

print('\nTotals by classification ($B):')
print((_lf.groupby('classification')['amount'].sum() / 1e9).round(3).to_string())

# A3 (audit 2026-08-26): the out_of_scope rows are CITY consumption taxes only. The School
# District's liquor-by-the-drink, cigarette, sales share, PILOTs and rideshare fees are out
# of scope too but are absent from this ledger entirely -- not in the City QCMR and not
# separately sourced. None is a tax on labor or capital, so no bundle changes.
print('')
print('NOTE: out_of_scope rows are CITY consumption taxes only. School District')
print('consumption taxes (liquor, cigarette, sales share) are excluded from the program')
print('but are not enumerated here.')

                                      name     $M classification        body vintage   status
                Wage & Earnings Tax (City) 2047.8          labor        city  FY2026 verified
                Wage & Earnings Tax (PICA)  731.3          labor        pica  FY2026 verified
                    Net Profits Tax (City)   39.1          labor        city  FY2026 verified
                    Net Profits Tax (PICA)   31.4          labor        pica  FY2026 verified
       Real property tax -- building share 1540.4        capital city+school  TY2026 verified
            Business Income & Receipts Tax  779.3        capital        city  FY2026 verified
                Realty Transfer Tax (City)  356.4        capital        city  FY2026 verified
              Business Use & Occupancy Tax  200.0        capital      school  FY2026 estimate
                         School Income Tax   71.8        capital      school  FY2025 estimate
Real property tax -- land share (ABSORBED)  602.2       abso

In [9]:
# Unverified lines, stated loudly. Spec section 3.
_est = [l for l in lines if l.status == 'estimate']
if _est:
    print('!' * 78)
    print(f'!! {len(_est)} LEDGER LINE(S) ARE ESTIMATES, NOT VERIFIED FIGURES')
    for l in _est:
        print(f'!!   {l.name}: ${l.amount/1e6:,.1f}M  [{l.vintage}]')
        print(f'!!     {l.note}')
    _est_total = sum(l.amount for l in _est)
    print(f'!! Combined: ${_est_total/1e9:.3f}B. They appear only in bundles B4 and B5.')
    print(f'!! Every kappa* for those bundles inherits this imprecision.')
    print('!' * 78)

_ok, _vintages = vintage_is_consistent(lines)
if not _ok:
    print(f'\nNOTE: computed lines span vintages {_vintages}. The property lines are '
          f'TY{TAX_YEAR} billed; the tax lines are {LEDGER_VINTAGE}; SIT is FY2025.')

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!! 2 LEDGER LINE(S) ARE ESTIMATES, NOT VERIFIED FIGURES
!!   School Income Tax: $71.8M  [FY2025]
!!     ESTIMATE: a secondary source (Controller analysis), not the School District's own adopted budget, so the status stays 'estimate'. Updated 2026-08-27 (audit A-2) from a $60.0M FY2023 floor ('more than $60 million', Revenue Dept. blog) that was 16.4% low and a vintage adrift -- the better figure was sitting in the same source already cited for the U&O line. Moves kappa*(B4) by about +0.0009; SIT is <1.5% of every bundle it appears in.
!!   Business Use & Occupancy Tax: $200.0M  [FY2026]
!!     ESTIMATE: rounded to the nearest $100M in a secondary source; the School District adopted budget is the primary document and was not machine-readable at build time. Separate levy from the property tax, so no overlap adjustment (spec 4.5). Vintage caveat (audit A-2, 2026-08-27): the Controller's own release presents the

## Section 5 — Bundles and the Headline kappa*

Six nested bundles, from the empty wiring check to the full program. `kappa*` is the uniform
capitalization rate at which each bundle exactly breaks even.

In [10]:
for name, b in BUNDLES.items():
    a = bundle_amounts(lines, name)
    print(f'{name}  {b["label"]:<32} target ${a["target"]/1e9:6.3f}B  '
          f'(abolished ${a["t_abolished"]/1e9:6.3f}B + land ${a["t_land"]/1e9:.3f}B)')

B0  None (wiring check)              target $ 0.602B  (abolished $ 0.000B + land $0.602B)
B1  Building tax only                target $ 2.143B  (abolished $ 1.540B + land $0.602B)
B2  Wage & Earnings only             target $ 3.381B  (abolished $ 2.779B + land $0.602B)
B3  Wage + entire property tax       target $ 4.922B  (abolished $ 4.320B + land $0.602B)
B4  B3 + BIRT + NPT + SIT            target $ 5.843B  (abolished $ 5.241B + land $0.602B)
B5  Full program (+ U&O + RTT)       target $ 6.400B  (abolished $ 5.798B + land $0.602B)


## Section 5b — Where kappa and G come from

The audit (2026-08-26) found that the ledger's `verified | estimate` discipline was applied
rigorously to the tax lines and not at all to the parameters that multiply them. These two
cells close that gap: every swept `kappa` is now one end of a published estimate, and the
road-rent levels come from a sibling model's committed run artifacts rather than from a
hand-set guess.

Evidence base and the mapping from published estimates to `kappa`:
`docs/KAPPA_CAPITALIZATION_EVIDENCE.md`. Two things to carry into any reading of the
`lit_*` columns:

- **`kappa` is still a swept parameter, not an estimate.** The headline result is `kappa*`,
  a break-even threshold that does not depend on any assumed `kappa`. The bounds bracket the
  `pot_`/`dividend_` columns only.
- **The `wage` family is anchored at the top by Jacob & Livas (2026)**: their open-city GE
  counterfactual for this exact tax in this exact city implies `kappa_wage = 1.056` —
  flow-on-flow, no discount-rate bridge, and a model-quantified *ceiling* rather than a mid
  estimate. The low bound is still transferred from a corporate-tax study, and nothing
  estimates the band's interior (realised capitalization under costly mobility). Largest
  family in every bundle from B2 up; still the weakest, no longer unanchored.


In [11]:
_kb = kappa_bounds_frame()
print('Published bounds on kappa, by family:')
print(_kb[['family', 'edge', 'kappa', 'basis', 'directness']].to_string(index=False))

_transferred = _kb[_kb.directness == 'transferred']
if len(_transferred):
    print()
    print('!' * 78)
    print(f'!! {len(_transferred)} KAPPA BOUND(S) ARE TRANSFERRED FROM A DIFFERENT TAX')
    for _, r in _transferred.iterrows():
        print(f'!!   {r["family"]}/{r["edge"]} = {r["kappa"]:.2f}  measured on: {r["setting"]}')
    print('!! Read every lit_low column containing that family accordingly.')
    print('!' * 78)

print()
for fam, b in KAPPA_BOUNDS.items():
    print(f'{fam}: [{b.low:.3f}, {b.high:.3f}]')
    print(f'  low  <- {b.low_source.citation[:96]}...')
    print(f'  high <- {b.high_source.citation[:96]}...')

Published bounds on kappa, by family:
  family edge  kappa                     basis  directness
building  low  0.000 landowner_incidence_share      direct
building high  1.006       capitalization_rate      direct
    wage  low  0.250 landowner_incidence_share transferred
    wage high  1.056      model_counterfactual      direct
   other  low  0.250 landowner_incidence_share      direct
   other high  0.300 landowner_incidence_share      direct

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!! 1 KAPPA BOUND(S) ARE TRANSFERRED FROM A DIFFERENT TAX
!!   wage/low = 0.25  measured on: US state corporate tax (NOT a wage tax)
!! Read every lit_low column containing that family accordingly.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

building: [0.000, 1.006]
  low  <- Loffler, Max and Sebastian Siegloch (2021), 'Welfare Effects of Property Taxation', IZA DP No. 1...
  high <- Coste, Jonah (2024), 'Capitalization of Propert

In [12]:
_rr = road_rent_frame()
print('Road/curb rent components (net-new annual):')
print(_rr[['g_level', 'name', 'amount', 'status']].to_string(index=False))
print()
for k, v in G_SCENARIOS.items():
    print(f'  G[{k}] = ${v/1e6:,.1f}M')

print()
print('Provenance: the congestion figures are p50 net revenue read from committed run')
print('artifacts in the sibling repo philly-cordon-pricing, not from its prose -- that')
print("repo's own audit (2026-04-22) found hand-quoted revenue figures that had drifted")
print('36-44% from its actual runs. Status is modeled_sibling, never verified.')
print()
print('Curb rent is carried at ZERO on purpose, not omitted:')
print(' ', _rr[_rr.key == 'curb_pricing'].iloc[0]['note'])

Road/curb rent components (net-new annual):
g_level                                                 name       amount          status
central   Center City cordon toll, $9/entry (NYC-calibrated)  83657804.40 modeled_sibling
   high Center City cordon toll, $25/entry (high deterrence) 175308213.52 modeled_sibling
  (all)               Curb parking rent (EXCLUDED, not zero)         0.00        excluded

  G[none] = $0.0M
  G[central] = $83.7M
  G[high] = $175.3M

Provenance: the congestion figures are p50 net revenue read from committed run
artifacts in the sibling repo philly-cordon-pricing, not from its prose -- that
repo's own audit (2026-04-22) found hand-quoted revenue figures that had drifted
36-44% from its actual runs. Status is modeled_sibling, never verified.

Curb rent is carried at ZERO on purpose, not omitted:
  Status-quo on-street revenue is ~$40-50M/yr, but Act 84 of 2012 already routes a $35M/yr minimum to the City General Fund and the residual to the School District, so

In [13]:
def kappa_table(surface, basis=('taxable',), phi=1.0, g_level='none', h=0.0,
                i=DEFAULT_I):
    rows = []
    for name in BUNDLES:
        a = bundle_amounts(lines, name)
        row = {'bundle': name, 'label': BUNDLES[name]['label']}
        for bas in basis:
            r0 = r0_by_case[(surface, bas, i)]
            row[bas] = kappa_star(a['target'], r0, a['t_abolished'],
                                  phi=phi, g=G_SCENARIOS[g_level], h=h)
        rows.append(row)
    return pd.DataFrame(rows)


print(f'kappa* at i={DEFAULT_I:.0%}, phi=1.0, no road rent, no haircut\n')
for surface in SURFACES:
    t = kappa_table(surface, basis=BASES)
    t = t.rename(columns={'taxable': f'{surface}/taxable', 'full': f'{surface}/full'})
    print(t.round(3).to_string(index=False))
    print()

kappa* at i=5%, phi=1.0, no road rent, no haircut

bundle                      label  opa/taxable  opa/full
    B0        None (wiring check)          NaN       NaN
    B1          Building tax only       -0.396    -0.438
    B2       Wage & Earnings only        0.226     0.203
    B3 Wage + entire property tax        0.502     0.487
    B4      B3 + BIRT + NPT + SIT        0.590     0.577
    B5 Full program (+ U&O + RTT)        0.629     0.618

bundle                      label  lycd/taxable  lycd/full
    B0        None (wiring check)           NaN        NaN
    B1          Building tax only        -1.211     -1.252
    B2       Wage & Earnings only        -0.225     -0.248
    B3 Wage + entire property tax         0.212      0.197
    B4      B3 + BIRT + NPT + SIT         0.350      0.338
    B5 Full program (+ U&O + RTT)         0.413      0.402



In [14]:
# The headline cell: central road-rent scenario, both surfaces, taxable basis.
_head = []
for name in BUNDLES:
    a = bundle_amounts(lines, name)
    row = {'bundle': name, 'label': BUNDLES[name]['label'],
           'target_$B': a['target'] / 1e9}
    for surface in SURFACES:
        r0 = r0_by_case[(surface, 'taxable', DEFAULT_I)]
        row[f'{surface}_G0'] = kappa_star(a['target'], r0, a['t_abolished'], phi=1.0)
        row[f'{surface}_Gcentral'] = kappa_star(
            a['target'], r0, a['t_abolished'], phi=1.0, g=G_SCENARIOS['central'])
    _head.append(row)
HEADLINE = pd.DataFrame(_head)
print('Break-even kappa by bundle (taxable basis, i=5%, phi=1.0)')
print(HEADLINE.round(3).to_string(index=False))
print('')
print('kappa* <= 0: pencils with no capitalization at all.')
print('kappa* > 1: needs super-ATCOR capitalization (EBCOR territory), not impossible.')

Break-even kappa by bundle (taxable basis, i=5%, phi=1.0)
bundle                      label  target_$B  opa_G0  opa_Gcentral  lycd_G0  lycd_Gcentral
    B0        None (wiring check)      0.602     NaN           NaN      NaN            NaN
    B1          Building tax only      2.143  -0.396        -0.451   -1.211         -1.265
    B2       Wage & Earnings only      3.381   0.226         0.196   -0.225         -0.256
    B3 Wage + entire property tax      4.922   0.502         0.483    0.212          0.192
    B4      B3 + BIRT + NPT + SIT      5.843   0.590         0.574    0.350          0.334
    B5 Full program (+ U&O + RTT)      6.400   0.629         0.615    0.413          0.398

kappa* <= 0: pencils with no capitalization at all.
kappa* > 1: needs super-ATCOR capitalization (EBCOR territory), not impossible.


In [15]:
# The two invariance results the spec asks the notebook to assert.
_r0 = r0_by_case[('opa', 'taxable', DEFAULT_I)]
_b0 = bundle_amounts(lines, 'B0')
_pot0 = 1.0 * _r0 - _b0['target']
assert abs(_pot0 / 2_151_145_182.0 - 1) < 0.01, f'B0 pot drift: ${_pot0:,.0f}'
assert abs(_pot0 / POPULATION - 1350.0) < 15.0

_b1 = bundle_amounts(lines, 'B1')
_pot1 = 1.0 * (_r0 + 1.0 * _b1['t_abolished']) - _b1['target']
assert np.isclose(_pot1, _pot0), 'B1 at kappa=1 must collapse onto B0'

print(f'B0 wiring check: pot ${_pot0/1e9:.3f}B = ${_pot0/POPULATION:,.0f}/resident. '
      f'Matches the LVT-UBI model.')
print(f'B1 invariance:   at kappa_building = 1 the pot is ${_pot1/1e9:.3f}B, identical to B0.')
print('  A structure tax under full capitalization is a land tax in disguise.')

B0 wiring check: pot $2.151B = $1,350/resident. Matches the LVT-UBI model.
B1 invariance:   at kappa_building = 1 the pot is $2.151B, identical to B0.
  A structure tax under full capitalization is a land tax in disguise.


## Section 6 — Sweep and Export

One row per (bundle x surface x basis x phi x i x road-rent level x haircut).

In [16]:
sweep = sweep_bundles(lines, r0_by_case, population=POPULATION,
                      phis=PHIS, haircuts=HAIRCUTS)
sweep['city'] = CITY_NAME
sweep['ledger_vintage'] = LEDGER_VINTAGE
sweep['tax_year'] = TAX_YEAR

_out = REPO_ROOT / 'analysis' / 'data' / f'{CITY_NAME}_ledger.csv'
sweep.to_csv(_out, index=False)
print(f'Sweep: {len(sweep):,} rows -> {_out.relative_to(REPO_ROOT)}')
print(f'Columns: {list(sweep.columns)}')

Sweep: 2,880 rows -> analysis\data\philadelphia_single_tax_ty2026_ledger.csv
Columns: ['bundle', 'bundle_label', 'surface', 'rent_basis', 'discount_rate', 'phi', 'g_level', 'g_amount', 'haircut', 'r0', 'target', 't_abolished', 't_land', 't_building', 't_wage', 't_other', 'kappa_star', 'pot_kappa_0', 'pot_kappa_0.25', 'pot_kappa_0.5', 'pot_kappa_0.75', 'pot_kappa_1', 'pot_lit_low', 'dividend_lit_low', 'pot_lit_high', 'dividend_lit_high', 'pot_atcor', 'dividend_atcor', 'city', 'ledger_vintage', 'tax_year']


In [17]:
_led = REPO_ROOT / 'analysis' / 'data' / f'{CITY_NAME}_lines.csv'
ledger_frame(lines).to_csv(_led, index=False)
print(f'Ledger: {len(lines)} lines -> {_led.relative_to(REPO_ROOT)}')

# A-1 (audit 2026-08-27): the tax lines ship their provenance beside the sweep; the kappa
# bounds did not, so a CSV-only reader met `dividend_lit_low` with nothing telling them the
# wage family's low bound is transferred from a different tax. Same companion-file
# convention, applied to the parameter that multiplies the lines.
_kbp = REPO_ROOT / 'analysis' / 'data' / f'{CITY_NAME}_kappa_bounds.csv'
kappa_bounds_frame().to_csv(_kbp, index=False)
print(f'Kappa bounds: {len(kappa_bounds_frame())} rows -> {_kbp.relative_to(REPO_ROOT)}')

# Dividend under the literature bounds and the ATCOR reference, central road rent.
# lit_low/lit_high are NOT a confidence interval -- see Section 5b.
_d = sweep[(sweep.surface == 'opa') & (sweep.rent_basis == 'taxable')
           & (sweep.discount_rate == DEFAULT_I) & (sweep.phi == 1.0)
           & (sweep.g_level == 'central') & (sweep.haircut == 0.0)]
_cols = ['bundle'] + [f'dividend_{s}' for s in KAPPA_SCENARIOS]
print('\nDividend per resident, OPA land, central road rent ($/yr):')
print(_d[_cols].round(0).to_string(index=False))

Ledger: 15 lines -> analysis\data\philadelphia_single_tax_ty2026_lines.csv
Kappa bounds: 6 rows -> analysis\data\philadelphia_single_tax_ty2026_kappa_bounds.csv

Dividend per resident, OPA land, central road rent ($/yr):
bundle  dividend_lit_low  dividend_lit_high  dividend_atcor
    B0            1403.0             1403.0          1403.0
    B1             436.0             1409.0          1403.0
    B2              94.0             1500.0          1403.0
    B3            -872.0             1506.0          1403.0
    B4           -1306.0             1135.0          1403.0
    B5           -1568.0              890.0          1403.0


In [18]:
# M4 (audit 2026-08-26): the ledger runs BILLED by default, but the QCMR tax lines are
# collections. Report both conventions rather than leaving the mix implicit.
from lvt.single_tax import DEFAULT_COLLECTION_RATE

_coll = build_ledger(LAND_TAX, BUILDING_TAX, vintage=LEDGER_VINTAGE,
                     collection_rate=DEFAULT_COLLECTION_RATE)
_rows = []
_r0d = r0_by_case[('opa', 'taxable', DEFAULT_I)]
for name in BUNDLES:
    a_b, a_c = bundle_amounts(lines, name), bundle_amounts(_coll, name)
    _rows.append({
        'bundle': name,
        'target_billed_$B': a_b['target'] / 1e9,
        'target_collected_$B': a_c['target'] / 1e9,
        'kappa*_billed': kappa_star(a_b['target'], _r0d, a_b['t_abolished'], phi=1.0),
        'kappa*_collected': kappa_star(a_c['target'], _r0d, a_c['t_abolished'], phi=1.0),
    })
print(f'Accrual sensitivity (OPA, i={DEFAULT_I:.0%}, phi=1.0, no road rent)')
print(f'Property collections/billings at TY2026 = {DEFAULT_COLLECTION_RATE:.4f}')
print(pd.DataFrame(_rows).round(3).to_string(index=False))
print('')
print('Billed is the default: it keeps the property lines identical to the LVT-UBI model')
print('and preserves the B0 wiring check. It is also the conservative direction -- since')
print('collections run below billings, a billed basis overstates what abolition costs the')
print('taxing bodies, and therefore overstates kappa*.')

Accrual sensitivity (OPA, i=5%, phi=1.0, no road rent)
Property collections/billings at TY2026 = 0.9452
bundle  target_billed_$B  target_collected_$B  kappa*_billed  kappa*_collected
    B0             0.602                0.569            NaN               NaN
    B1             2.143                2.025         -0.396            -0.500
    B2             3.381                3.348          0.226             0.214
    B3             4.922                4.804          0.502             0.484
    B4             5.843                5.726          0.590             0.576
    B5             6.400                6.282          0.629             0.618

Billed is the default: it keeps the property lines identical to the LVT-UBI model
and preserves the B0 wiring check. It is also the conservative direction -- since
collections run below billings, a billed basis overstates what abolition costs the
taxing bodies, and therefore overstates kappa*.


## Section 7 — Grounding the Haircut

The haircut `h` stands in for ordinary delinquency and for the negative-rent surrender tail at
full capture (Limitation 17 of the LVT-UBI guide: land whose rent exceeds what any user will
pay is abandoned to the City). This section is evidence for which `h` is plausible, not a
surrender model — that is its own future notebook.

In [19]:
import requests

_q = ("SELECT opa_number, total_due, num_years_owed, oldest_year_owed "
      "FROM real_estate_tax_delinquencies")
try:
    _r = requests.get('https://phl.carto.com/api/v2/sql',
                      params={'q': _q, 'format': 'csv'}, timeout=180)
    _r.raise_for_status()
    from io import StringIO
    delq = pd.read_csv(StringIO(_r.text), dtype={'opa_number': str})
    DELQ_OK = True
except Exception as e:
    print(f'Delinquency fetch failed ({e}); skipping Section 7 evidence.')
    DELQ_OK = False

if DELQ_OK:
    delq['_key'] = delq['opa_number'].astype(str).str.strip().str.lstrip('0')
    gdf['_key'] = gdf['parcel_number'].str.lstrip('0')
    m = gdf[['_key', 'land_opa', 'land_lycd', 'taxable_total']].merge(
        delq[['_key', 'total_due', 'num_years_owed']], on='_key', how='left')
    m['delinquent'] = m['total_due'].notna()
    m['chronic'] = m['num_years_owed'].fillna(0) >= 3

    print(f'Delinquent parcels:        {m.delinquent.sum():,} '
          f'({m.delinquent.mean()*100:.1f}% of {len(m):,})')
    print(f'  total due:               ${m.total_due.sum()/1e6:,.0f}M')
    print(f'  3+ years owed (chronic): {m.chronic.sum():,} '
          f'({m.chronic.mean()*100:.1f}%)')
    for s in SURFACES:
        share = m.loc[m.chronic, f'land_{s}'].sum() / m[f'land_{s}'].sum()
        print(f'  chronic share of {s.upper()} land value: {share*100:.1f}%')
    gdf = gdf.drop(columns=['_key'])

Delinquent parcels:        52,044 (8.9% of 583,204)
  total due:               $379M
  3+ years owed (chronic): 32,511 (5.6%)
  chronic share of OPA land value: 2.7%
  chronic share of LYCD land value: 4.0%


In [20]:
if DELQ_OK:
    _chronic_share = m.loc[m.chronic, 'land_opa'].sum() / m['land_opa'].sum()
    print(f'\nReading: chronic delinquency covers {_chronic_share*100:.1f}% of OPA land '
          f'value today, under a levy of {COMBINED_RATE*100:.4f}% of assessed value.')
    print(f'Full capture charges roughly {(DEFAULT_I + COMBINED_RATE)/COMBINED_RATE:.1f}x '
          f'that on land, so h = 0 is not a defensible central case.')
    print(f'The sweep carries h in {HAIRCUTS}; h = 0.05 is the repo\'s known ordinary '
          f'delinquency wedge, and higher h is the surrender tail.')


Reading: chronic delinquency covers 2.7% of OPA land value today, under a levy of 1.3998% of assessed value.
Full capture charges roughly 4.6x that on land, so h = 0 is not a defensible central case.
The sweep carries h in (0.0, 0.05, 0.1, 0.15); h = 0.05 is the repo's known ordinary delinquency wedge, and higher h is the surrender tail.


## Section 8 — Charts

In [21]:
# Chart 1: kappa* heatmap, bundle x scenario.
_scen = []
for surface in SURFACES:
    for g in ('none', 'central'):
        for phi in PHIS:
            _scen.append((surface, g, phi))
_mat = np.zeros((len(BUNDLES) - 1, len(_scen)))
_bnames = [b for b in BUNDLES if b != 'B0']
for r, bname in enumerate(_bnames):
    a = bundle_amounts(lines, bname)
    for c, (surface, g, phi) in enumerate(_scen):
        _mat[r, c] = kappa_star(a['target'],
                                r0_by_case[(surface, 'taxable', DEFAULT_I)],
                                a['t_abolished'], phi=phi, g=G_SCENARIOS[g])

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(np.clip(_mat, 0, 1.5), cmap='RdYlGn_r', vmin=0, vmax=1.5, aspect='auto')
ax.set_xticks(range(len(_scen)))
ax.set_xticklabels([f'{s.upper()}\nG={g}\nphi={p}' for s, g, p in _scen], fontsize=8)
ax.set_yticks(range(len(_bnames)))
ax.set_yticklabels([f'{b} — {BUNDLES[b]["label"]}' for b in _bnames], fontsize=9)
for r in range(_mat.shape[0]):
    for c in range(_mat.shape[1]):
        v = _mat[r, c]
        ax.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='white' if (v > 1.0 or v < 0.15) else 'black')
ax.set_title(f'Break-even capitalization kappa* by bundle  (i={DEFAULT_I:.0%}, '
             f'taxable basis, TY{TAX_YEAR})', fontsize=11)
cb = fig.colorbar(im, ax=ax, shrink=0.85)
cb.set_label('kappa*  (<=0 pencils outright, >1 needs super-ATCOR)', fontsize=9)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'kappa_star_heatmap.png', dpi=150)
plt.close(fig)
print('kappa_star_heatmap.png')

kappa_star_heatmap.png


In [22]:
# Chart 2: dividend per resident vs kappa, by bundle.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
_ks = np.linspace(0, 1, 51)
for ax, surface in zip(axes, SURFACES):
    r0 = r0_by_case[(surface, 'taxable', DEFAULT_I)]
    for bname in BUNDLES:
        a = bundle_amounts(lines, bname)
        y = [(1.0 * (r0 + k * a['t_abolished']) + G_SCENARIOS['central']
              - a['target']) / POPULATION for k in _ks]
        ax.plot(_ks, y, label=f'{bname} {BUNDLES[bname]["label"]}', lw=1.8)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f'{surface.upper()} land surface')
    ax.set_xlabel('kappa (share of abolished tax reappearing as land rent)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Dividend per resident ($/yr)')
axes[1].legend(fontsize=7.5, loc='upper left')
fig.suptitle(f'Residual dividend vs capitalization  (i={DEFAULT_I:.0%}, phi=1.0, '
             f'central road rent)', fontsize=11)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'dividend_vs_kappa.png', dpi=150)
plt.close(fig)
print('dividend_vs_kappa.png')

dividend_vs_kappa.png


In [23]:
# Chart 3: ledger waterfall -- what the full program must replace, against supply.
_lf2 = ledger_frame(lines)
_ab = _lf2[_lf2.classification.isin(['labor', 'capital'])].sort_values(
    'amount', ascending=False)
_names = list(_ab['name']) + ['Land tax (absorbed)']
_vals = list(_ab['amount'] / 1e9) + [LAND_TAX / 1e9]

fig, ax = plt.subplots(figsize=(11, 6.0))
_left = np.concatenate([[0], np.cumsum(_vals)[:-1]])
_colors = ['#c0504d'] * (len(_vals) - 1) + ['#7f7f7f']
ax.barh(range(len(_vals)), _vals, left=_left, color=_colors, edgecolor='white')
for k, (n, v, l) in enumerate(zip(_names, _vals, _left)):
    if v >= 0.30:   # label inside only where the bar is wide enough to hold it
        ax.text(l + v / 2, k, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='white')
    else:
        ax.text(l + v + 0.06, k, f'{v:.2f}', ha='left', va='center', fontsize=8,
                color='#333333')
ax.set_yticks(range(len(_names)))
ax.set_yticklabels(_names, fontsize=9)
ax.invert_yaxis()
_total = sum(_vals)
for surface, color in zip(SURFACES, ['#4f81bd', '#9bbb59']):
    r0 = r0_by_case[(surface, 'taxable', DEFAULT_I)] / 1e9
    ax.axvline(r0, color=color, lw=2.2, ls='--',
               label=f'{surface.upper()} land rent R0 = ${r0:.2f}B')
    ax.axvline(r0 + G_SCENARIOS['central'] / 1e9, color=color, lw=1.2, ls=':',
               label=f'{surface.upper()} + central road rent')
ax.axvline(_total, color='k', lw=2.0,
           label=f'Full program target = ${_total:.2f}B')
ax.set_xlabel('$B per year (cumulative)')
ax.set_xlim(0, _total * 1.08)
ax.set_title(f'What the full program must replace, against what land rent supplies '
             f'(TY{TAX_YEAR}, i={DEFAULT_I:.0%})', fontsize=11)
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
          frameon=False)
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(REPORT_DIR / 'ledger_waterfall.png', dpi=150)
plt.close(fig)
print('ledger_waterfall.png')
print(f'\nCharts in {REPORT_DIR.relative_to(REPO_ROOT)}')

ledger_waterfall.png

Charts in analysis\reports\philadelphia_single_tax


## Section 9 — Summary

In [24]:
_b3 = bundle_amounts(lines, 'B3')
_b5 = bundle_amounts(lines, 'B5')
print(f'Full program (B5) target:        ${_b5["target"]/1e9:.3f}B')
print(f'OPA land rent at i={DEFAULT_I:.0%}:          '
      f'${r0_by_case[("opa","taxable",DEFAULT_I)]/1e9:.3f}B')
print(f'LYCD land rent at i={DEFAULT_I:.0%}:         '
      f'${r0_by_case[("lycd","taxable",DEFAULT_I)]/1e9:.3f}B')
print()
for name in ('B3', 'B5'):
    a = bundle_amounts(lines, name)
    for surface in SURFACES:
        r0 = r0_by_case[(surface, 'taxable', DEFAULT_I)]
        k0 = kappa_star(a['target'], r0, a['t_abolished'], phi=1.0)
        kc = kappa_star(a['target'], r0, a['t_abolished'], phi=1.0,
                        g=G_SCENARIOS['central'])
        print(f'{name} on {surface.upper():5}: kappa* = {k0:.3f} (no road rent), '
              f'{kc:.3f} (central road rent)')

Full program (B5) target:        $6.400B
OPA land rent at i=5%:          $2.753B
LYCD land rent at i=5%:         $4.008B

B3 on OPA  : kappa* = 0.502 (no road rent), 0.483 (central road rent)
B3 on LYCD : kappa* = 0.212 (no road rent), 0.192 (central road rent)
B5 on OPA  : kappa* = 0.629 (no road rent), 0.615 (central road rent)
B5 on LYCD : kappa* = 0.413 (no road rent), 0.398 (central road rent)


**B2 pencils on LYCD land at the central parameters, and only there.** At `i = 5%`,
`phi = 1.0` and a small haircut, LYCD site rent covers the entire Wage & Earnings Tax with
`kappa* < 0` — no capitalization required at all. That result is **not robust**: it fails in
every `i = 3%` cell, and at `phi = 0.85` it is knife-edge (`kappa*` about −0.01 with no
haircut, turning positive by `h = 5%`). Across the sensitivity grid checked in the audit it
held in fewer than half the cells. State it with its operating conditions or not at all.

**The result.** The bundle that matters is B3 — abolish the Wage & Earnings Tax and the
entire property tax on structures, absorb the land tax, fund it all from site rent. On the
LYCD land surface it needs only a small amount of capitalization; on OPA's land surface, which
carries the 20% default ratio on ~45% of improved parcels, it needs several times more. That
spread *is* the finding: whether Philadelphia's land rent can carry its two biggest taxes
depends on which land surface you believe, and the two published surfaces disagree by 46%.

**The full program does not pencil at zero capitalization, and whether published
estimates could carry it depends on the surface.** Read B5's `kappa*` off the table above
(at `G = 0`, `phi = 1`, `h = 0` it is about 0.63 on OPA land and 0.41 on LYCD). An earlier
version of this cell said B5 "requires capitalization well above what any published
estimate of ATCOR-style incidence would support, on either surface"; the 2026-08-27 audit
(M-2) struck that as contradicted by this notebook's own new bounds. Weighting B5's
families by their shares of `t_abolished` (wage 49%, building 27%, other 24%) and using
published *point estimates* only — no theory ceilings — gives `kappa` of about 0.45–0.46
on the Coste side of the building disagreement, which clears LYCD but not OPA; on the
Löffler side it gives about 0.18–0.20 and clears neither. So the honest statement is
conditional on both the land surface and on which building-capitalization study one
believes. Reporting that conditionality plainly, rather than a flat negative, is the
point of building the ledger.

**What would move these numbers most, in order:** the `kappa` band above; the land surface (46% between OPA and LYCD,
larger than any other axis here); `phi`, since stopping at 0.85 for the assessment reasons in
the LVT-UBI guide costs 15% of supply directly; the two estimate lines, if B4/B5 are the
bundles of interest; and `i`, which scales R0 but — unlike in the LVT-UBI model, where it was
a pure scale knob — does move `kappa*` here, because the Target side is denominated in dollars
of tax revenue that do not scale with `i`.

**What the literature says about `kappa`, added 2026-08-27.** Every swept `kappa` is now
one end of a published estimate (`docs/KAPPA_CAPITALIZATION_EVIDENCE.md`), and the honest
summary is that the evidence does not pin it. The `building` family is bracketed
`[0.000, 1.006]` — Loffler & Siegloch (2021) find German property taxes pass fully to
tenants with landowners bearing ~none, while Coste (2024) finds Philadelphia's own
abatement is capitalized at **100.6%** (95% CI 86.0–115.2%) into new-construction prices.
Both are well identified; they disagree. The `wage` family, the largest in every bundle
from B2 up, is anchored at the top since Tier 2 by **Jacob & Livas (2026)**: their
open-city counterfactual for this exact tax implies `kappa_wage = 1.056` — a
model-quantified ceiling that internalises the commuter-export margin. Its low bound is
still transferred from a corporate-tax study, and no estimate exists for the interior of
the band (realised capitalization under costly mobility). So `kappa*` should still be
read against a range, not a point, and B3's spread between surfaces remains the larger
of the two uncertainties.

**An artifact of the bounds, and what it is not.** At `lit_high` and `phi = 1`,
`kappa_building = 1.006 > 1` and B1's dividend comes out $5.80/resident *above* B0's.
That is the arithmetic signature of a `kappa` above 1 — not evidence for one. The margin
is `T_building x 0.006 / population`, the direct consequence of the assumed bound; Coste's
95% CI (86.0–115.2%) spans 1.0 and his own reading is *full* capitalization; and at
`phi = 0.85` the sign reverses to $140/resident *below* B0. An earlier version of this
cell called it "EBCOR in miniature, demonstrated" — the 2026-08-27 audit (M-1) struck
that for holding in exactly half the export's cells while stating none of its operating
conditions, which is the standard the B2 paragraph above sets. EBCOR remains a reason
`kappa > 1` is not impossible; nothing here demonstrates it. On how demanding that region
is: it needs `theta*(1 + MEB/revenue) > 1`, which at the sourced landowner shares
(0.25–0.30) and mainstream excess-burden estimates means `theta` near 0.64–0.81.

**Not modelled:** every behavioural response. See spec section 10 and
`docs/LVT_UBI_GUIDE.md` limitations.